# Vietnamese Emotion Classification - Hybrid Two-Stage Training

**🌟 Google Colab Ready!** This notebook is optimized for Google Colab with GPU acceleration.

## 🎯 Hybrid Training Strategy

This notebook implements a **two-stage training approach** that combines the benefits of:
- **Stage 1**: Training on original (clean) data for strong baseline representations
- **Stage 2**: Fine-tuning on augmented data for improved robustness and generalization

### Why Hybrid Training?

| Approach | Pros | Cons |
|----------|------|------|
| Original Data Only | Clean signal, good precision | May overfit, less robust |
| Augmented Data Only | More diverse, regularization | May introduce noise |
| **Hybrid (This)** | Best of both worlds | Slightly longer training |

### Training Flow

```
Stage 1: Original Data (Higher LR)
    ↓ Learn clean representations
    ↓ ~10 epochs until convergence
    ↓
Stage 2: Augmented Data (Lower LR)
    ↓ Fine-tune with regularization
    ↓ ~5 epochs with strict early stopping
    ↓
Final Model: Robust & Generalized
```

---

## 🚀 Advanced Techniques

- ✅ **Layer-wise Learning Rate Decay (LLRD)**
- ✅ **Slanted Triangular Learning Rate (STLR)**
- ✅ **Two-Stage Training with LR Reduction**
- ✅ **Early Stopping**

---

**Let's get started!**


## 🔧 Google Colab Setup


In [ ]:
# Mount Google Drive to access training data
from google.colab import drive
drive.mount('/content/drive')

print("✓ Google Drive mounted successfully")
print("✓ Your data should be in: /content/drive/MyDrive/thesis/data")

# Check for TPU
import os
try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    TPU_AVAILABLE = True
    print("✓ TPU detected and configured")
except ImportError:
    TPU_AVAILABLE = False
    print("ℹ️  TPU not available, will use GPU/CPU")


## 📋 Configuration


In [ ]:
# ============================================================================
# HYBRID TRAINING CONFIGURATION
# ============================================================================

# Random seed for reproducibility
SEED = 42

# Data paths
DATA_DIR = '/content/drive/MyDrive/thesis/data'
MODEL_SAVE_PATH = '/content/drive/MyDrive/thesis/emotion_classifier_hybrid'

# Model selection
MODEL_NAME = 'uitnlp/CafeBERT'

# Batch size settings
PER_DEVICE_TRAIN_BATCH_SIZE = 24
PER_DEVICE_EVAL_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 4

# ============================================================================
# STAGE 1: Training on Original Data
# ============================================================================
STAGE1_EPOCHS = 15          # Train until near-convergence
STAGE1_BASE_LR = 2e-5       # Standard learning rate
STAGE1_WARMUP_RATIO = 0.1   # 10% warmup
STAGE1_EARLY_STOPPING = 4   # Patience for early stopping

# ============================================================================
# STAGE 2: Fine-tuning on Augmented Data
# ============================================================================
STAGE2_EPOCHS = 8           # Fewer epochs (regularization only)
STAGE2_BASE_LR = 5e-6       # 4x lower than Stage 1
STAGE2_WARMUP_RATIO = 0.05  # Shorter warmup (model already trained)
STAGE2_EARLY_STOPPING = 3   # Stricter early stopping

# Common settings
LR_DECAY_FACTOR = 0.95
CLASSIFIER_LR_MULTIPLIER = 10.0
STLR_RATIO = 32
WEIGHT_DECAY = 0.01

# Text preprocessing
USE_UNDERTHESEA_TOKENIZER = True

print("✓ Hybrid Training Configuration Loaded")
print(f"\n{'='*60}")
print("STAGE 1: Original Data Training")
print(f"{'='*60}")
print(f"  Epochs: {STAGE1_EPOCHS}")
print(f"  Base LR: {STAGE1_BASE_LR:.2e}")
print(f"  Warmup: {STAGE1_WARMUP_RATIO*100:.0f}%")
print(f"  Early Stopping Patience: {STAGE1_EARLY_STOPPING}")

print(f"\n{'='*60}")
print("STAGE 2: Augmented Data Fine-tuning")
print(f"{'='*60}")
print(f"  Epochs: {STAGE2_EPOCHS}")
print(f"  Base LR: {STAGE2_BASE_LR:.2e} (4x lower)")
print(f"  Warmup: {STAGE2_WARMUP_RATIO*100:.0f}%")
print(f"  Early Stopping Patience: {STAGE2_EARLY_STOPPING}")


## 1. Install & Import Libraries


In [ ]:
%pip install -q transformers datasets torch scikit-learn pandas numpy matplotlib seaborn openpyxl underthesea wandb


In [ ]:
import os
import gc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                              f1_score, precision_recall_fscore_support)
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
import json
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(SEED)
torch.manual_seed(SEED)

# Check device
USE_TPU = False

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✓ Using device: {device}")
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
    print(f"✓ Using device: {device} (Metal Performance Shaders)")
else:
    device = torch.device('cpu')
    print(f"✓ Using device: {device}")
    print("  ⚠️  Warning: No GPU detected. Training will be slower.")

print("✓ All libraries imported successfully")


## 2. Weights & Biases Setup (Optional)


In [ ]:
import wandb
from google.colab import userdata

try:
    wandb_api_key = userdata.get('WANDB_API_KEY')
    wandb.login(key=wandb_api_key)
    print("✓ Successfully logged into wandb")
except Exception as e:
    print(f"⚠️  wandb login failed: {e}")
    print("   Continuing without wandb tracking...")

# Initialize wandb
try:
    wandb.init(
        project="emotion-classification-hybrid",
        name=f"cafebert-hybrid-training",
        config={
            "model": MODEL_NAME,
            "stage1_lr": STAGE1_BASE_LR,
            "stage2_lr": STAGE2_BASE_LR,
            "stage1_epochs": STAGE1_EPOCHS,
            "stage2_epochs": STAGE2_EPOCHS,
            "experiment_type": "hybrid_training"
        }
    )
    print("✓ wandb initialized")
except:
    print("⚠️  wandb initialization skipped")


## 3. LLRD + STLR Implementation


In [ ]:
def get_optimizer_grouped_parameters(model, base_lr=2e-5, lr_decay_factor=0.9,
                                    weight_decay=0.01, classifier_lr_multiplier=10.0):
    """Create parameter groups with layer-wise learning rate decay."""

    no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
    optimizer_grouped_parameters = []

    # Get model type
    if hasattr(model, 'roberta'):
        encoder = model.roberta
        model_type = 'roberta'
    elif hasattr(model, 'bert'):
        encoder = model.bert
        model_type = 'bert'
    else:
        raise ValueError("Model type not supported for LLRD")

    num_layers = len(encoder.encoder.layer)

    print(f"\n{'='*80}")
    print("LAYER-WISE LEARNING RATE DECAY (LLRD)")
    print(f"{'='*80}")
    print(f"Model: {model_type} | Layers: {num_layers} | Base LR: {base_lr:.2e} | Decay: {lr_decay_factor}")
    print("\nLearning rates by layer:")

    # 1. Classifier head (highest LR)
    classifier_lr = base_lr * classifier_lr_multiplier
    optimizer_grouped_parameters.extend([
        {'params': [p for n, p in model.classifier.named_parameters() if not any(nd in n for nd in no_decay)],
         'lr': classifier_lr, 'weight_decay': weight_decay},
        {'params': [p for n, p in model.classifier.named_parameters() if any(nd in n for nd in no_decay)],
         'lr': classifier_lr, 'weight_decay': 0.0}
    ])
    print(f"  Classifier head: {classifier_lr:.2e}")

    # 2. Encoder layers (top to bottom with decay)
    for layer_idx in range(num_layers - 1, -1, -1):
        layer = encoder.encoder.layer[layer_idx]
        lr = base_lr * (lr_decay_factor ** (num_layers - 1 - layer_idx))

        optimizer_grouped_parameters.extend([
            {'params': [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
             'lr': lr, 'weight_decay': weight_decay},
            {'params': [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)],
             'lr': lr, 'weight_decay': 0.0}
        ])

        if layer_idx % 3 == 0:
            print(f"  Layer {layer_idx}: {lr:.2e}")

    # 3. Embeddings (lowest LR)
    embedding_lr = base_lr * (lr_decay_factor ** num_layers)
    optimizer_grouped_parameters.extend([
        {'params': [p for n, p in encoder.embeddings.named_parameters() if not any(nd in n for nd in no_decay)],
         'lr': embedding_lr, 'weight_decay': weight_decay},
        {'params': [p for n, p in encoder.embeddings.named_parameters() if any(nd in n for nd in no_decay)],
         'lr': embedding_lr, 'weight_decay': 0.0}
    ])
    print(f"  Embeddings: {embedding_lr:.2e}")
    print(f"{'='*80}\n")

    return optimizer_grouped_parameters


def get_slanted_triangular_scheduler(optimizer, num_training_steps, cut_frac=0.1, ratio=32):
    """Slanted Triangular Learning Rate Scheduler (STLR) from ULMFiT."""
    cut = int(num_training_steps * cut_frac)
    
    def lr_lambda(current_step):
        if current_step < cut:
            p = current_step / cut
            return (1 + p * (ratio - 1)) / ratio
        else:
            p = (current_step - cut) / (num_training_steps - cut)
            return (1 + (1 - p) * (ratio - 1)) / ratio
    
    return LambdaLR(optimizer, lr_lambda)

print("✓ LLRD function defined")
print("✓ STLR scheduler function defined")


In [ ]:
class AdvancedTrainer(Trainer):
    """Advanced Trainer with LLRD + STLR Support."""

    def __init__(self, *args, llrd_config=None, **kwargs):
        self.llrd_config = llrd_config or {}
        super().__init__(*args, **kwargs)

    def create_optimizer(self):
        """Create optimizer with LLRD parameter groups."""
        if self.optimizer is None:
            optimizer_grouped_parameters = get_optimizer_grouped_parameters(
                self.model,
                base_lr=self.llrd_config.get('base_lr', 2e-5),
                lr_decay_factor=self.llrd_config.get('lr_decay_factor', 0.9),
                weight_decay=self.args.weight_decay,
                classifier_lr_multiplier=self.llrd_config.get('classifier_lr_multiplier', 10.0)
            )

            self.optimizer = AdamW(optimizer_grouped_parameters, betas=(0.9, 0.999), eps=1e-8)

        return self.optimizer

    def create_scheduler(self, num_training_steps: int, optimizer=None):
        """Create STLR scheduler combined with LLRD."""
        if self.lr_scheduler is None:
            cut_frac = self.args.warmup_ratio
            ratio = self.llrd_config.get('stlr_ratio', 32)
            
            self.lr_scheduler = get_slanted_triangular_scheduler(
                optimizer=self.optimizer if optimizer is None else optimizer,
                num_training_steps=num_training_steps,
                cut_frac=cut_frac,
                ratio=ratio
            )
            
            cut_steps = int(num_training_steps * cut_frac)
            print(f"\n{'='*60}")
            print("SLANTED TRIANGULAR LEARNING RATE (STLR)")
            print(f"{'='*60}")
            print(f"  Total steps: {num_training_steps}")
            print(f"  Warmup (cut): {cut_steps} steps ({cut_frac*100:.0f}%)")
            print(f"  Ratio: {ratio} (min LR = max LR / {ratio})")
            print(f"  ⚠️  Scheduler steps after EVERY batch")
            print(f"{'='*60}\n")

        return self.lr_scheduler

print("✓ AdvancedTrainer class defined (LLRD + STLR)")


## 4. Load and Prepare Data


In [ ]:
# Underthesea word segmentation for Vietnamese
_underthesea_available = False
if USE_UNDERTHESEA_TOKENIZER:
    try:
        from underthesea import word_tokenize as uts_word_tokenize
        _underthesea_available = True
        print("✓ Underthesea word tokenizer loaded")
    except ImportError:
        print("⚠️  Underthesea not available, falling back to simple tokenization")

def tokenize_vietnamese(text):
    """Tokenize Vietnamese text using underthesea word segmentation."""
    if text is None:
        return ""
    t = str(text).strip()
    if _underthesea_available:
        t = uts_word_tokenize(t, format='text')
    return t

print(f"✓ Vietnamese text preprocessing ready")


In [ ]:
# Create model save directory
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
print(f"✓ Model save directory ready: {MODEL_SAVE_PATH}")

# ============================================================================
# LOAD DATASETS
# ============================================================================
# Original data (for Stage 1)
train_original_df = pd.read_csv(os.path.join(DATA_DIR, 'processed/train_org_processed.csv'))

# Augmented data (for Stage 2) - use balanced/augmented dataset
train_augmented_df = pd.read_csv(os.path.join(DATA_DIR, 'processed/train_2500_final.csv'))

# Validation and Test sets (same for both stages)
val_df = pd.read_csv(os.path.join(DATA_DIR, 'processed/val_processed.csv'))
test_df = pd.read_csv(os.path.join(DATA_DIR, 'processed/test_processed.csv'))

print(f"\n✓ Datasets loaded from Google Drive")
print(f"  Stage 1 (Original): {train_original_df.shape[0]} samples")
print(f"  Stage 2 (Augmented): {train_augmented_df.shape[0]} samples")
print(f"  Validation: {val_df.shape[0]} samples")
print(f"  Test: {test_df.shape[0]} samples")


In [ ]:
# Standardize column names
print("Original data columns:", train_original_df.columns.tolist())
print("Augmented data columns:", train_augmented_df.columns.tolist())

# Process original data
if 'Sentence_clean' in train_original_df.columns:
    train_original_df = train_original_df[['Sentence_clean', 'emotion_vn']].copy()
    train_original_df.columns = ['text', 'label']
elif 'Sentence' in train_original_df.columns:
    train_original_df = train_original_df[['Sentence', 'emotion_vn']].copy()
    train_original_df.columns = ['text', 'label']

# Process augmented data
if 'Sentence_clean' in train_augmented_df.columns and 'emotion_vn' in train_augmented_df.columns:
    train_augmented_df = train_augmented_df[['Sentence_clean', 'emotion_vn']].copy()
    train_augmented_df.columns = ['text', 'label']
elif 'Sentence' in train_augmented_df.columns:
    train_augmented_df = train_augmented_df[['emotion_vn', 'Sentence_clean']].copy()
    train_augmented_df.columns = ['label', 'text']

# Process val/test
val_df.columns = ['text', 'label']
test_df.columns = ['text', 'label']

print("\n✓ Column names standardized")


In [ ]:
# Clean and tokenize all datasets
all_dfs = [
    ('Original Train', train_original_df),
    ('Augmented Train', train_augmented_df),
    ('Validation', val_df),
    ('Test', test_df)
]

for name, df in all_dfs:
    df.dropna(subset=['text', 'label'], inplace=True)
    df['text'] = df['text'].astype(str).str.strip()
    df['label'] = df['label'].astype(str).str.strip()
    
    if USE_UNDERTHESEA_TOKENIZER:
        df['text'] = df['text'].apply(tokenize_vietnamese)
    
    print(f"✓ Processed {name}: {df.shape[0]} samples")

print("\n✓ All datasets cleaned and tokenized")


In [ ]:
# Create label mappings
all_labels = pd.concat([train_original_df['label'], train_augmented_df['label']]).unique()
unique_labels = sorted(all_labels)
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}
num_classes = len(unique_labels)

# Add numeric labels
train_original_df['labels'] = train_original_df['label'].map(label2id)
train_augmented_df['labels'] = train_augmented_df['label'].map(label2id)
val_df['labels'] = val_df['label'].map(label2id)
test_df['labels'] = test_df['label'].map(label2id)

print(f"\n✓ Label mappings created ({num_classes} classes)")
print(f"  Labels: {list(label2id.keys())}")


## 5. Analyze Class Distribution


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original data distribution
original_counts = train_original_df['label'].value_counts().sort_index()
axes[0].bar(original_counts.index, original_counts.values, color='steelblue', edgecolor='black')
axes[0].set_title('Stage 1: Original Data Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Augmented data distribution
augmented_counts = train_augmented_df['label'].value_counts().sort_index()
axes[1].bar(augmented_counts.index, augmented_counts.values, color='coral', edgecolor='black')
axes[1].set_title('Stage 2: Augmented Data Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Emotion')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"\nOriginal data imbalance ratio: {original_counts.max() / original_counts.min():.1f}:1")
print(f"Augmented data imbalance ratio: {augmented_counts.max() / augmented_counts.min():.1f}:1")


## 6. Tokenize Data & Load Model


In [ ]:
# Convert to HuggingFace Dataset format
train_original_dataset = Dataset.from_pandas(train_original_df[['text', 'labels']].reset_index(drop=True))
train_augmented_dataset = Dataset.from_pandas(train_augmented_df[['text', 'labels']].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[['text', 'labels']].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[['text', 'labels']].reset_index(drop=True))

print(f"✓ Datasets converted to HuggingFace format")

# Load tokenizer
print(f"\nLoading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✓ Tokenizer loaded")

# Tokenize
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=256)

train_original_tokenized = train_original_dataset.map(tokenize_function, batched=True)
train_augmented_tokenized = train_augmented_dataset.map(tokenize_function, batched=True)
val_tokenized = val_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

print(f"\n✓ Tokenized Stage 1 (Original): {len(train_original_tokenized)} samples")
print(f"✓ Tokenized Stage 2 (Augmented): {len(train_augmented_tokenized)} samples")
print(f"✓ Tokenized Validation: {len(val_tokenized)} samples")
print(f"✓ Tokenized Test: {len(test_tokenized)} samples")


In [ ]:
# Define metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_weighted = f1_score(labels, predictions, average='weighted')

    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted
    }

print("✓ Metrics function defined")


## 7. 🚀 STAGE 1: Train on Original Data

**Goal:** Learn clean, high-quality representations from original data


In [ ]:
print("="*80)
print("🚀 STAGE 1: Training on ORIGINAL Data")
print("="*80)
print(f"\nDataset: {len(train_original_tokenized)} samples")
print(f"Epochs: {STAGE1_EPOCHS}")
print(f"Base LR: {STAGE1_BASE_LR:.2e}")
print(f"Early Stopping Patience: {STAGE1_EARLY_STOPPING}")
print("="*80)


In [ ]:
# Initialize fresh model for Stage 1
model_stage1 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_classes,
    id2label=id2label,
    label2id=label2id
)
model_stage1 = model_stage1.to(device)

print(f"✓ Model loaded: {sum(p.numel() for p in model_stage1.parameters()):,} parameters")


In [ ]:
# Stage 1 Training Arguments
training_args_stage1 = TrainingArguments(
    output_dir='./results_stage1_original',
    num_train_epochs=STAGE1_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=STAGE1_BASE_LR,
    warmup_ratio=STAGE1_WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    logging_dir='./logs_stage1',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_weighted',
    greater_is_better=True,
    save_total_limit=2,
    fp16=(device.type == 'cuda'),
    dataloader_num_workers=0,
    seed=SEED,
    report_to='wandb' if 'wandb' in dir() else 'none',
)

# LLRD config for Stage 1
llrd_config_stage1 = {
    'base_lr': STAGE1_BASE_LR,
    'lr_decay_factor': LR_DECAY_FACTOR,
    'classifier_lr_multiplier': CLASSIFIER_LR_MULTIPLIER,
    'stlr_ratio': STLR_RATIO
}

# Create Stage 1 trainer
trainer_stage1 = AdvancedTrainer(
    model=model_stage1,
    args=training_args_stage1,
    train_dataset=train_original_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=STAGE1_EARLY_STOPPING)],
    llrd_config=llrd_config_stage1
)

print("✓ Stage 1 Trainer created")


In [ ]:
# Train Stage 1
print("\n🚀 Starting Stage 1 training...")
stage1_result = trainer_stage1.train()

# Evaluate Stage 1
print("\n" + "="*60)
print("📊 STAGE 1 RESULTS (Original Data)")
print("="*60)

stage1_val_eval = trainer_stage1.evaluate(val_tokenized)
stage1_test_eval = trainer_stage1.evaluate(test_tokenized)

print(f"\nValidation Results:")
print(f"  Accuracy:    {stage1_val_eval['eval_accuracy']:.4f}")
print(f"  F1 Macro:    {stage1_val_eval['eval_f1_macro']:.4f}")
print(f"  F1 Weighted: {stage1_val_eval['eval_f1_weighted']:.4f}")

print(f"\nTest Results:")
print(f"  Accuracy:    {stage1_test_eval['eval_accuracy']:.4f}")
print(f"  F1 Macro:    {stage1_test_eval['eval_f1_macro']:.4f}")
print(f"  F1 Weighted: {stage1_test_eval['eval_f1_weighted']:.4f}")

print(f"\nTraining Time: {stage1_result.metrics['train_runtime']:.2f}s")
print("="*60)


## 8. 🚀 STAGE 2: Fine-tune on Augmented Data

**Goal:** Improve robustness using augmented data as regularization
 

In [ ]:
print("\n" + "="*80)
print("🚀 STAGE 2: Fine-tuning on AUGMENTED Data")
print("="*80)
print(f"\nDataset: {len(train_augmented_tokenized)} samples")
print(f"Epochs: {STAGE2_EPOCHS}")
print(f"Base LR: {STAGE2_BASE_LR:.2e} (4x lower than Stage 1)")
print(f"Early Stopping Patience: {STAGE2_EARLY_STOPPING}")
print("="*80)

# Clear memory
gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()
elif device.type == 'mps':
    torch.mps.empty_cache()

print("\n✓ Memory cleared for Stage 2")


In [ ]:
# Get the best model from Stage 1 (already loaded due to load_best_model_at_end=True)
model_stage2 = trainer_stage1.model

# Stage 2 Training Arguments (LOWER learning rate!)
training_args_stage2 = TrainingArguments(
    output_dir='./results_stage2_hybrid',
    num_train_epochs=STAGE2_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=STAGE2_BASE_LR,  # 4x lower than Stage 1
    warmup_ratio=STAGE2_WARMUP_RATIO,  # Shorter warmup
    weight_decay=WEIGHT_DECAY,
    logging_dir='./logs_stage2',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_weighted',
    greater_is_better=True,
    save_total_limit=2,
    fp16=(device.type == 'cuda'),
    dataloader_num_workers=0,
    seed=SEED,
    report_to='wandb' if 'wandb' in dir() else 'none',
)

# LLRD config for Stage 2 (lower base LR)
llrd_config_stage2 = {
    'base_lr': STAGE2_BASE_LR,
    'lr_decay_factor': LR_DECAY_FACTOR,
    'classifier_lr_multiplier': CLASSIFIER_LR_MULTIPLIER,
    'stlr_ratio': STLR_RATIO
}

# Create Stage 2 trainer
trainer_stage2 = AdvancedTrainer(
    model=model_stage2,  # Continue from Stage 1 model!
    args=training_args_stage2,
    train_dataset=train_augmented_tokenized,  # Augmented data
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=STAGE2_EARLY_STOPPING)],
    llrd_config=llrd_config_stage2
)

print("✓ Stage 2 Trainer created (continuing from Stage 1 model)")


In [ ]:
# Train Stage 2
print("\n🚀 Starting Stage 2 fine-tuning...")
stage2_result = trainer_stage2.train()

# Evaluate Stage 2
print("\n" + "="*60)
print("📊 STAGE 2 RESULTS (Augmented Data Fine-tuning)")
print("="*60)

stage2_val_eval = trainer_stage2.evaluate(val_tokenized)
stage2_test_eval = trainer_stage2.evaluate(test_tokenized)

print(f"\nValidation Results:")
print(f"  Accuracy:    {stage2_val_eval['eval_accuracy']:.4f}")
print(f"  F1 Macro:    {stage2_val_eval['eval_f1_macro']:.4f}")
print(f"  F1 Weighted: {stage2_val_eval['eval_f1_weighted']:.4f}")

print(f"\nTest Results:")
print(f"  Accuracy:    {stage2_test_eval['eval_accuracy']:.4f}")
print(f"  F1 Macro:    {stage2_test_eval['eval_f1_macro']:.4f}")
print(f"  F1 Weighted: {stage2_test_eval['eval_f1_weighted']:.4f}")

print(f"\nTraining Time: {stage2_result.metrics['train_runtime']:.2f}s")
print("="*60)


## 9. 📊 Final Comparison: Stage 1 vs Stage 2


In [ ]:
print("\n" + "="*80)
print("📊 HYBRID TRAINING COMPARISON")
print("="*80)

# Create comparison table
comparison_data = {
    'Metric': ['Accuracy', 'F1 Macro', 'F1 Weighted'],
    'Stage 1 (Original)': [
        f"{stage1_test_eval['eval_accuracy']:.4f}",
        f"{stage1_test_eval['eval_f1_macro']:.4f}",
        f"{stage1_test_eval['eval_f1_weighted']:.4f}"
    ],
    'Stage 2 (Hybrid)': [
        f"{stage2_test_eval['eval_accuracy']:.4f}",
        f"{stage2_test_eval['eval_f1_macro']:.4f}",
        f"{stage2_test_eval['eval_f1_weighted']:.4f}"
    ],
    'Improvement': [
        f"{(stage2_test_eval['eval_accuracy'] - stage1_test_eval['eval_accuracy'])*100:+.2f}%",
        f"{(stage2_test_eval['eval_f1_macro'] - stage1_test_eval['eval_f1_macro'])*100:+.2f}%",
        f"{(stage2_test_eval['eval_f1_weighted'] - stage1_test_eval['eval_f1_weighted'])*100:+.2f}%"
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\nTest Set Performance:")
print(comparison_df.to_string(index=False))

# Determine winner
stage2_better = stage2_test_eval['eval_f1_weighted'] > stage1_test_eval['eval_f1_weighted']
print(f"\n{'='*80}")
if stage2_better:
    improvement = (stage2_test_eval['eval_f1_weighted'] - stage1_test_eval['eval_f1_weighted']) * 100
    print(f"✅ HYBRID TRAINING IMPROVED PERFORMANCE by {improvement:.2f}%")
    print(f"   Stage 2 (Hybrid) is the better model!")
else:
    print(f"⚠️  Stage 1 (Original) performed better")
    print(f"   Consider adjusting Stage 2 parameters")
print("="*80)


In [ ]:
# Visualize comparison
metrics = ['Accuracy', 'F1 Macro', 'F1 Weighted']
stage1_scores = [stage1_test_eval['eval_accuracy'], stage1_test_eval['eval_f1_macro'], stage1_test_eval['eval_f1_weighted']]
stage2_scores = [stage2_test_eval['eval_accuracy'], stage2_test_eval['eval_f1_macro'], stage2_test_eval['eval_f1_weighted']]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, stage1_scores, width, label='Stage 1 (Original)', color='steelblue')
bars2 = ax.bar(x + width/2, stage2_scores, width, label='Stage 2 (Hybrid)', color='coral')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Hybrid Training: Stage 1 vs Stage 2 Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=10)

for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()


## 10. Save Final Model


In [ ]:
# Save the best model (Stage 2 hybrid model)
final_model = trainer_stage2.model

# Save model and tokenizer
final_model.save_pretrained(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)

# Save label mappings
with open(os.path.join(MODEL_SAVE_PATH, 'label_mappings.json'), 'w') as f:
    json.dump({'label2id': label2id, 'id2label': id2label}, f, ensure_ascii=False, indent=2)

# Save training summary
training_summary = {
    'model_name': MODEL_NAME,
    'training_type': 'hybrid_two_stage',
    'stage1': {
        'data': 'original',
        'epochs': STAGE1_EPOCHS,
        'base_lr': STAGE1_BASE_LR,
        'test_accuracy': stage1_test_eval['eval_accuracy'],
        'test_f1_macro': stage1_test_eval['eval_f1_macro'],
        'test_f1_weighted': stage1_test_eval['eval_f1_weighted']
    },
    'stage2': {
        'data': 'augmented',
        'epochs': STAGE2_EPOCHS,
        'base_lr': STAGE2_BASE_LR,
        'test_accuracy': stage2_test_eval['eval_accuracy'],
        'test_f1_macro': stage2_test_eval['eval_f1_macro'],
        'test_f1_weighted': stage2_test_eval['eval_f1_weighted']
    }
}

with open(os.path.join(MODEL_SAVE_PATH, 'training_summary.json'), 'w') as f:
    json.dump(training_summary, f, indent=2)

print(f"\n✅ Hybrid model saved to: {MODEL_SAVE_PATH}")
print(f"   - Model weights")
print(f"   - Tokenizer")
print(f"   - Label mappings")
print(f"   - Training summary")


## 11. Final Classification Report


In [ ]:
# Get predictions on test set
predictions = trainer_stage2.predict(test_tokenized)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

# Classification report
print("\n" + "="*80)
print("FINAL CLASSIFICATION REPORT (Hybrid Model)")
print("="*80)
print(classification_report(labels, preds, target_names=list(id2label.values())))


In [ ]:
# Confusion matrix
cm = confusion_matrix(labels, preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=list(id2label.values()),
            yticklabels=list(id2label.values()))
plt.title('Confusion Matrix - Hybrid Model', fontsize=14, fontweight='bold')
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 12. Cleanup


In [ ]:
# Finish wandb run
try:
    wandb.finish()
    print("✓ wandb run finished")
except:
    pass

# Clear memory
gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()

print("\n" + "="*80)
print("✅ HYBRID TRAINING COMPLETE!")
print("="*80)
print(f"\nFinal Model: {MODEL_SAVE_PATH}")
print(f"\nBest Test Results:")
print(f"  Accuracy:    {stage2_test_eval['eval_accuracy']:.4f}")
print(f"  F1 Macro:    {stage2_test_eval['eval_f1_macro']:.4f}")
print(f"  F1 Weighted: {stage2_test_eval['eval_f1_weighted']:.4f}")
print("="*80)
